In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
import importlib

# Ellenőrizzük, hogy a statsmodels elérhető-e a trendline-hoz
statsmodels_available = importlib.util.find_spec("statsmodels") is not None

# Adatok beolvasása
df = pd.read_csv('student_lifestyle_performance_dataset.csv')

# Kimeneti mappa
os.makedirs('interactive_plots', exist_ok=True)

# Egyedi színek és sablon
custom_template = {
    'layout': {
        'font': {'family': 'Arial, sans-serif', 'size': 12},
        'title': {'font': {'size': 18, 'family': 'Arial', 'weight': 'bold'}},
        'plot_bgcolor': '#f8f9fa',
        'paper_bgcolor': '#ffffff',
        'xaxis': {'showgrid': True, 'gridcolor': '#e9ecef', 'title_font': {'size': 13}},
        'yaxis': {'showgrid': True, 'gridcolor': '#e9ecef', 'title_font': {'size': 13}},
    }
}
px.defaults.template = 'plotly_white'  # alap sablon

# --- 1. CGPA eloszlás (hisztogram + doboz) ---
fig1 = px.histogram(df, x='CGPA', nbins=25, marginal='box', 
                    title='📊 CGPA eloszlása a hallgatók körében',
                    labels={'CGPA': 'CGPA', 'count': 'Gyakoriság'},
                    color_discrete_sequence=['#2c7fb8'])
fig1.update_layout(height=550, title_x=0.5, **custom_template['layout'])
fig1.update_traces(marker_line_width=1, marker_line_color='white')
fig1.write_html('interactive_plots/plot1_cgpa_dist.html')

# --- 2. Stressz vs tanulás (scatter, színezés lakóhely szerint) ---
fig2 = px.scatter(df, x='Study_Hours_per_Day', y='Stress_Level_1_to_10',
                  title='😰 Tanulási órák és stressz kapcsolata',
                  labels={'Study_Hours_per_Day': '📚 Napi tanulási órák', 
                          'Stress_Level_1_to_10': '😩 Stressz szint (1-10)'},
                  color='Residence', opacity=0.65,
                  color_discrete_sequence=px.colors.qualitative.Set2,
                  trendline='ols' if statsmodels_available else None)
fig2.update_layout(height=550, title_x=0.5, **custom_template['layout'])
fig2.write_html('interactive_plots/plot2_stress_study.html')

# --- 3. Átlagos CGPA szakonként (oszlopdiagram) ---
branch_cgpa = df.groupby('Branch')['CGPA'].mean().reset_index().sort_values('CGPA', ascending=False)
fig3 = px.bar(branch_cgpa, x='Branch', y='CGPA', 
              title='🏆 Átlagos CGPA szakonként',
              labels={'CGPA': 'Átlagos CGPA', 'Branch': 'Szak'},
              color='CGPA', color_continuous_scale='Tealgrn',
              text_auto='.2f')
fig3.update_traces(marker_line_width=1, marker_line_color='white')
fig3.update_layout(height=550, title_x=0.5, **custom_template['layout'])
fig3.write_html('interactive_plots/plot3_cgpa_by_branch.html')

# --- 4. CGPA diéta szerint (violin plot szebb, mint a boxplot) ---
fig4 = px.violin(df, x='Diet_Type', y='CGPA', box=True, points='all',
                 title='🥗 CGPA megoszlása diéta típus szerint',
                 labels={'Diet_Type': 'Diéta', 'CGPA': 'CGPA'},
                 color='Diet_Type', color_discrete_sequence=px.colors.qualitative.Pastel)
fig4.update_layout(height=550, title_x=0.5, **custom_template['layout'])
fig4.write_html('interactive_plots/plot4_cgpa_diet.html')

# --- 5. Korrelációs hőtérkép (javított, nem csúszik el a szöveg) ---
import plotly.graph_objects as go

numeric_cols = ['Age', 'Study_Hours_per_Day', 'Sleep_Hours', 'Screen_Time_Hours',
                'Gym_Hours_per_Week', 'Attendance_Percentage', 'Stress_Level_1_to_10',
                'Internal_Marks', 'CGPA']
corr = df[numeric_cols].corr()

# Oszlopok/rövidítések a szebb megjelenésért
col_names = ['Kor', 'Tanulás', 'Alvás', 'Képernyő', 'Edzés', 
             'Részvétel', 'Stressz', 'Belső jegy', 'CGPA']

fig5 = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=col_names,
    y=col_names,
    colorscale='RdBu_r',
    zmin=-1, zmax=1,
    text=corr.round(2).values,
    texttemplate='%{text}',
    textfont={"size": 11, "color": "black"},
    hoverongaps=False,
    hovertemplate='%{x}<br>%{y}<br>Korreláció: %{z:.2f}<extra></extra>'
))

fig5.update_layout(
    title='🔗 Korrelációs mátrix – Hogyan függ össze minden?',
    title_x=0.5,
    height=650,
    width=None,
    font=dict(family='Arial, sans-serif', size=12),
    xaxis=dict(title='', tickangle=-45, tickfont=dict(size=10)),
    yaxis=dict(title='', tickfont=dict(size=10)),
    template='plotly_white'
)

fig5.write_html('interactive_plots/plot5_correlation_heatmap.html')

# --- 6. Tanulási órák vs belső jegyek (szín: szak) ---
fig6 = px.scatter(df, x='Study_Hours_per_Day', y='Internal_Marks',
                  title='✍️ Tanulási órák és belső jegyek kapcsolata',
                  labels={'Study_Hours_per_Day': '📖 Napi tanulás (óra)', 
                          'Internal_Marks': '📝 Belső jegyek (max 100)'},
                  color='Branch', opacity=0.7,
                  color_discrete_sequence=px.colors.qualitative.G10,
                  trendline='ols' if statsmodels_available else None)
fig6.update_layout(height=550, title_x=0.5, **custom_template['layout'])
fig6.write_html('interactive_plots/plot6_study_internal.html')

# --- 7. Részvételi arány eloszlása (hisztogram + doboz) ---
fig7 = px.histogram(df, x='Attendance_Percentage', nbins=20, marginal='box',
                    title='🏫 Részvételi százalék eloszlása',
                    labels={'Attendance_Percentage': 'Részvétel (%)', 'count': 'Gyakoriság'},
                    color_discrete_sequence=['#2ca02c'])
fig7.update_layout(height=550, title_x=0.5, **custom_template['layout'])
fig7.update_traces(marker_line_width=1, marker_line_color='white')
fig7.write_html('interactive_plots/plot7_attendance_dist.html')

# --- 8. Stressz lakóhely szerint (oszlop + hibasáv) ---
residence_stress = df.groupby('Residence')['Stress_Level_1_to_10'].agg(['mean', 'std']).reset_index()
residence_stress.columns = ['Residence', 'mean_stress', 'std_stress']
fig8 = px.bar(residence_stress, x='Residence', y='mean_stress', 
              error_y='std_stress',
              title='🏠 Átlagos stressz szint lakóhely típus szerint',
              labels={'Residence': 'Lakóhely', 'mean_stress': 'Átlagos stressz (1-10)'},
              color='Residence', color_discrete_sequence=px.colors.sequential.Blues_r,
              text_auto='.2f')
fig8.update_layout(height=550, title_x=0.5, **custom_template['layout'])
fig8.write_html('interactive_plots/plot8_stress_residence.html')

# --- 9. Edzésórák vs CGPA (szín: diéta) ---
fig9 = px.scatter(df, x='Gym_Hours_per_Week', y='CGPA',
                  title='💪 Heti edzésórák és CGPA kapcsolata',
                  labels={'Gym_Hours_per_Week': '🏋️ Heti edzés (óra)', 
                          'CGPA': 'CGPA'},
                  color='Diet_Type', opacity=0.7,
                  color_discrete_sequence=px.colors.qualitative.Set1,
                  trendline='ols' if statsmodels_available else None)
fig9.update_layout(height=550, title_x=0.5, **custom_template['layout'])
fig9.write_html('interactive_plots/plot9_gym_cgpa.html')

# --- 10. Képernyőidő vs alvás (szín: stresszszint) ---
fig10 = px.scatter(df, x='Screen_Time_Hours', y='Sleep_Hours',
                   title='📱 Képernyőidő és alvásórák – a digitális élet ára',
                   labels={'Screen_Time_Hours': '📺 Képernyőidő (óra/nap)', 
                           'Sleep_Hours': '😴 Alvás (óra/nap)'},
                   color='Stress_Level_1_to_10', color_continuous_scale='Plasma',
                   trendline='ols' if statsmodels_available else None,
                   opacity=0.7)
fig10.update_layout(height=550, title_x=0.5, **custom_template['layout'])
fig10.write_html('interactive_plots/plot10_screen_sleep.html')

print("✅ 10 interaktív diagram elkészült az 'interactive_plots' mappában.")
if not statsmodels_available:
    print("⚠️ A statsmodels modul nem található. A trendvonalak kimaradtak. Telepítsd: pip install statsmodels")
else:
    print("📈 Trendvonalak (OLS) bekerültek a scatter plotokba.")

✅ 10 interaktív diagram elkészült az 'interactive_plots' mappában.
📈 Trendvonalak (OLS) bekerültek a scatter plotokba.
